## Importe as bibliotecas

In [ ]:
# Configuração inicial
import pandas as pd
import numpy as np
from scipy import stats as st
import seaborn as sns
from matplotlib import pyplot as plt
import re
import pytest

## Definição das configurações de visualização.

In [ ]:
# Variável Global para aleatoriedade
SEED = 42

# Ajuste das quantidade das colunas exibidas
pd.set_option('display.max_columns', 100)

# Estilo e tamanho padrão dos gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 7)

## Funções Auxiliares

In [ ]:
def calcular_bins(dados):
    """
    Parâmetros
    ----------
    dados : array-like (list, numpy.ndarray, pandas.Series)
        Conjunto de dados numéricos que será analisado.

    Retorna
    -------
    dict
        Dicionário contendo a quantidade de bins sugerida por três métodos:
        - "sqrt_rule": Regra da raiz quadrada (simples, rápida, baseada em √n)
        - "sturges_rule": Regra de Sturges (log2(n) + 1, boa para distribuições normais)
        - "freedman_diaconis": Regra de Freedman–Diaconis (baseada no IQR, mais robusta a outliers)

    Observações
    -----------
    - √n tende a funcionar bem em distribuições grandes e variadas.
    - Sturges é mais conservadora, útil para distribuições próximas à normal.
    - Freedman–Diaconis se adapta melhor a dados assimétricos e com outliers.

    Exemplo
    -------
    bins = calcular_bins(df['sales'])
    plt.hist(df['sales'], bins=bins['freedman_diaconis'])
    """

    # Calcula o tamanho da amostra (número de observações)
    n = len(dados)
    # Garante que os dados estão em formato NumPy array
    dados = np.asarray(dados)
    # 1. Raiz quadrada do tamanho da amostra
    bins_sqrt = int(np.sqrt(n))

    # 2. Regra de Sturges: log2(n) + 1
    bins_sturges = int(np.log2(n) + 1)

    # 3. Regra de Freedman–Diaconis
    q75, q25 = np.percentile(dados, [75, 25])   # Calcula os quartis
    iqr = q75 - q25                             # Intervalo interquartil (IQR)
    
    # Calcula a largura ideal do bin (bin_width) pela regra de FD
    bin_width = 2 * iqr * n ** (-1/3) if iqr > 0 else 1

    # Calcula o número de bins como o intervalo total / largura
    bins_fd = int((dados.max() - dados.min()) / bin_width) if bin_width > 0 else bins_sturges

    # Retorna os resultados como dicionário
    return {
        "sqrt_rule": bins_sqrt,
        "sturges_rule": bins_sturges,
        "freedman_diaconis": bins_fd
    }


In [ ]:
def outliers_iqr(series):
    """
    Identifica outliers em uma série numérica utilizando o método do Intervalo Interquartil (IQR).

    Parâmetros
    ----------
    series : pandas.Series
        Série numérica (coluna do DataFrame) onde serão identificados os outliers.

    Retorna
    -------
    mask : pandas.Series (boolean)
        Máscara booleana que indica True para os valores considerados outliers.
    lower : float
        Limite inferior para considerar um valor como não outlier.
    upper : float
        Limite superior para considerar um valor como não outlier.

    Exemplo
    -------
    mask, lower, upper = outliers_iqr(df['sales'])
    df[mask]  # retorna apenas os registros considerados outliers
    """

    # Calcula o primeiro quartil (25%)
    q1 = series.quantile(0.25)

    # Calcula o terceiro quartil (75%)
    q3 = series.quantile(0.75)

    # Calcula o intervalo interquartil (IQR = Q3 - Q1)
    iqr = q3 - q1

    # Define o limite inferior (Q1 - 1.5 * IQR)
    lower = q1 - 1.5 * iqr

    # Define o limite superior (Q3 + 1.5 * IQR)
    upper = q3 + 1.5 * iqr

    # Cria uma máscara booleana que marca os valores abaixo do limite inferior ou acima do superior
    mask = (series < lower) | (series > upper)

    # Retorna a máscara e os limites calculados
    return mask, lower, upper


In [ ]:
def outliers_zscore(series):
    """
    Identifica outliers em uma série numérica utilizando o método do Z-score.

    Parâmetros
    ----------
    series : pandas.Series
        Série numérica (coluna do DataFrame) onde serão identificados os outliers.

    Retorna
    -------
    mask : pandas.Series (boolean)
        Máscara booleana que indica True para os valores considerados outliers.
        (valores cujo Z-score absoluto é maior que 3).
    z : pandas.Series (float)
        Série contendo o valor padronizado (Z-score) de cada elemento.

    Observações
    -----------
    - O Z-score mede quantos desvios padrão cada valor está distante da média.
    - Por convenção, valores com |Z| > 3 são considerados outliers.
    - É utilizado ddof=0 para calcular o desvio padrão populacional (consistente com o Z-score).

    Exemplo
    -------
    mask, z = outliers_zscore(df['sales'])
    df[mask]  # retorna apenas os registros considerados outliers
    """

    # Remove valores ausentes (NaN) para cálculo da média e do desvio padrão
    s = series.dropna()

    # Calcula a média da série
    mean = s.mean()

    # Calcula o desvio padrão populacional (ddof=0)
    std = s.std(ddof=0)  # se usássemos ddof=1, seria o desvio padrão amostral

    # Calcula o Z-score: (valor - média) / desvio padrão
    # Se o desvio padrão for 0, usa 1.0 para evitar divisão por zero
    z = (series - mean) / (std if std != 0 else 1.0)

    # Cria uma máscara booleana: True para valores cujo |Z| > 3 (outliers)
    mask = z.abs() > 3

    # Retorna a máscara e a série de Z-scores
    return mask, z


In [ ]:
def safe_mode(series):
    """
    Calcula a moda (valor mais frequente) de uma série numérica ou categórica de forma segura.

    Parâmetros
    ----------
    series : pandas.Series
        Coluna de um DataFrame (numérica ou categórica) para a qual se deseja calcular a moda.

    Retorna
    -------
    valor : qualquer tipo ou float
        Retorna o valor mais frequente (moda) da série.
        Caso não exista uma moda única (ex.: múltiplos valores empatados),
        retorna NaN para evitar erro.

    Observações
    -----------
    - Usa a função mode da biblioteca `statistics`.
    - O método `.dropna()` é aplicado para ignorar valores ausentes (NaN).
    - Em cenários com múltiplas modas, `statistics.mode()` gera `StatisticsError`.
      Esse erro é tratado e a função retorna `np.nan`.

    Exemplo
    -------
    >>> safe_mode(pd.Series([1, 2, 2, 3, 4]))
    2

    >>> safe_mode(pd.Series([1, 1, 2, 2]))  # múltiplas modas
    nan
    """

    # Remove valores ausentes antes do cálculo da moda
    try:
        return series.dropna().mode()
    except StatisticsError:
        # Se houver empate (múltiplas modas), retorna NaN
        return np.nan


In [ ]:
def safe_series_to_int(serie):
    """
    Esta função fará a conversão de FLOAT to INT de toda coluna se for segura.
    Conversão é segura quando não se perdem informações decimais em nenhum registro da coluna.
    Recebe objeto series e retorna o objeto series convertido para INT se for aprovada a conversão.
    Caso não seja aprovada, retorna "Observe a linha X"
    """
    if np.array_equal(serie, serie.astype(int)):
        return serie.astype(int)
    else:
        for idx, item in serie.items():
            if item != int(item):
                return f"Observe a linha {idx}"

        return f"Observe a linha {index}"


# Criando uma função de teste relativa à nova função criada
def test_safe_series_to_int():
    # Caso 1: todos os valores são conversões seguras (floats "inteiros")
    serie1 = pd.Series([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0])
    resultado1 = safe_series_to_int(serie1)
    esperado1 = serie1.astype(int)
    assert resultado1.equals(esperado1), f"Caso 1 falhou: {resultado1}"

    # Caso 2: índice customizado, valor quebrado (5.35) na posição 4 -> índice real 104
    serie2 = pd.Series(
        [1.0, 2.0, 3.0, 4.0, 5.35, 6.0, 7.0, 8.0, 9.0, 10.0],
        index=[100, 101, 102, 103, 104, 105, 106, 107, 108, 109]
    )
    resultado2 = safe_series_to_int(serie2)
    assert resultado2 == "Observe a linha 104", f"Caso 2 falhou: {resultado2}"

    # Caso 3: valores negativos, todos seguros
    serie3 = pd.Series([-1.0, -2.0, -3.0, -4.0, -5.0, -6.0, -7.0, -8.0, -9.0, -10.0])
    resultado3 = safe_series_to_int(serie3)
    esperado3 = serie3.astype(int)
    assert resultado3.equals(esperado3), f"Caso 3 falhou: {resultado3}"

    print("Todos os testes passaram!")
    return True

print(test_safe_series_to_int())



In [93]:
def separate_name(name):
    """
    Esta função será usada para separar os respectivos nomes dos jogos
    quando houver duplicidade de registro apenas no nome
    Exemplos: 
    - NBA Live 06 (All region sales)
    - Project Gotham Racing (JP weekly sales)

    Ela retorna uma tupla contendo (name, case) nesta ordem
    """
    location = name.find("(") 

    first_group = name[0:location]
    second_group = name[location:len(name)]

    return (first_group.strip(), second_group.strip())

def test_separate_name():
    # Caso 1: nome com sufixo "All region sales"
    entrada1 = "NBA Live 06 (All region sales)"
    resultado1 = separate_name(entrada1)
    esperado1 = ("NBA Live 06", "(All region sales)")
    assert resultado1 == esperado1, f"Caso 1 falhou: {resultado1}"

    # Caso 2: nome com sufixo "JP weekly sales"
    entrada2 = "Project Gotham Racing (JP weekly sales)"
    resultado2 = separate_name(entrada2)
    esperado2 = ("Project Gotham Racing", "(JP weekly sales)")
    assert resultado2 == esperado2, f"Caso 2 falhou: {resultado2}"

    # Caso 3: nome com sufixo "JP sales"
    entrada3 = "The Godfather (JP sales)"
    resultado3 = separate_name(entrada3)
    esperado3 = ("The Godfather", "(JP sales)")
    assert resultado3 == esperado3, f"Caso 3 falhou: {resultado3}"

    print("Todos os testes passaram!")
    return True

print(test_separate_name())
    

Todos os testes passaram!
True


## Leitura dos dados

In [ ]:
df = pd.read_csv('games.csv')
df.head()

In [ ]:
df.columns

In [ ]:
# Padronizando os nomes das colunas para snake_case:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
    .str.replace(r"[^a-z0-9_]", "", regex=True)
)

df.head(1)

In [ ]:
df.info()

## Análise Inicial e Limpeza de Dados:

Vamos prosseguir com uma observação inicial do conjunto de dados e tomar decisões importantes, em cada coluna, para melhorar a qualidade dos dados que estamos trabalhando. 
As decisões de cada coluna serão tomadas com base naquilo que será o melhor para o nosso objetivo final: planejar campanhas publicitárias que se convertam em resultados para a empresa Ice.

### Coluna 0: `name`

In [ ]:
column = df["name"]

print("Verificando a presença de duplicados:")
print(column.duplicated().sum())
print("-"*42)
print("Verificando a presença de ausentes:")
print(column.isna().sum())
print("-"*42)
# Variável Categórica
print("Valores observados:")
print(f"Total de valores diferentes: {column.nunique()}")
print(f"Top 5 valores mais frequentes: {column.value_counts().head()}")
print("-"*42)
print(column.unique().tolist())

Preencheremos os valores ausentes com "unknown" porque são jogos de 1993 que não interferem na nossa análise.

In [ ]:
df["name"] = df["name"].fillna("unknown")
print("Verificando a presença de ausentes:")
print(df["name"].isna().sum())

Destrinchando um pouco mais os valores desta coluna, vamos entender o comportamento dos duplicados mais a fundo. Observemos o "FIFA 14", um jogo bastante repetido como visto acima.

In [ ]:
fifa14_only = df[df["name"] == "FIFA 14"]
fifa14_only

In [ ]:
# Vamos contar quantas repetições possuem do par NAME-PLATFORM
possible_duplicated = df.duplicated(subset=["name","platform"], keep=False)
df[possible_duplicated]

In [ ]:
implicit_example = df["name"].str.contains("Medal of Honor: European Assault")
df[implicit_example]

#### Duplicados da Coluna `name`

Com os códigos acima, percebemos dois comportamentos interessantes que devemos resolver nessa coluna:

1. Possíveis Duplicados Explícitos: jogos com o mesmo nome, plataforma e ano de lançamento, mostrados em diferentes registros de vendas totais.

2. Possíveis Duplicados Implícitos: jogos que apresentam o mesmo nome, plataforma e ano de lançamento, mas estão divididos por "categorias" impressas nos seus nomes: `(All Region sales)`, `(JP weekly sales)`, `(Old all region sales)`, etc. 

Utilizaremos duas abordagens diferentes para lidar com eles antes de analisar a próxima coluna.

In [79]:
# Localizando todos os explícitos, agora incluindo ano de lançamento
# Máscara boleana para identificá-los
possible_duplicated = df.duplicated(subset=["name","platform","year_of_release"], keep=False)
df[possible_duplicated]

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
659,unknown,GEN,1993,unknown,1.78,0.53,0.00,0.08,68.97,7.125,unknown
14244,unknown,GEN,1993,unknown,0.00,0.00,0.03,0.00,68.97,7.125,unknown


Os jogos "unknown" não apresentam informações suficientes para afirmarmos que é o mesmo jogo. Enquanto isso, o jogo "Madden NFL 13" parece apontar para o mesmo jogo, e portanto condensaremos o seu dado de venda `eu_sales` no registro de índice `604`, e excluiremos o registro `16230`.

In [ ]:
# Atualizando o valor do primeiro registro
df.iloc[604,5] = df.iloc[604,5] + df.iloc[16230,5].astype(float)
print(f"Novo valor da célula observada: {float(df.iloc[604,5]):.3}")

# Apagando o registro da linha 16230
df = df.drop(labels=16230)

Agora vamos observar alguns duplicados implícitos que aparentam apontar para outros registros:

In [83]:
implicits = df["name"].str.contains(r'\([^)]*sales[^)]*\)', regex=True, na=False)
df_implicits = df[implicits]
df_implicits

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
788,Project Gotham Racing (JP weekly sales),XB,2002,ACTION,1.54,0.44,0.04,0.07,68.97,7.125,unknown
920,Medal of Honor: European Assault (All Region s...,PS2,2005,SHOOTER,0.89,0.69,0.09,0.21,68.97,7.125,unknown
1123,NBA Live 06 (All region sales),PS2,2005,SPORTS,1.44,0.15,0.00,0.05,68.97,7.125,unknown
1136,Tony Hawk's American Wasteland (Old all region...,PS2,2005,SPORTS,0.80,0.63,0.01,0.19,68.97,7.125,unknown
1324,Tony Hawk's American Wasteland (Weekly america...,PS2,2005,SPORTS,1.38,0.05,0.00,0.02,68.97,7.125,unknown
...,...,...,...,...,...,...,...,...,...,...,...
16426,Rondo of Swords (jp sales),DS,2007,ROLE-PLAYING,0.00,0.00,0.01,0.00,68.97,7.125,unknown
16506,The Godfather (JP sales),X360,2006,ACTION,0.00,0.00,0.01,0.00,68.97,7.125,unknown
16616,National Geographic Panda (JP sales),DS,2008,SIMULATION,0.00,0.00,0.01,0.00,68.97,7.125,unknown
16656,Imagine Figure Skater (JP sales),DS,2007,SPORTS,0.00,0.00,0.01,0.00,68.97,7.125,unknown


vamos criar uma função auxiliar para separar todos os `name` em uma tupla com (name, case) de forma que possamos saber exatamente de qual jogo se trata, e o que o registro representa.

#### Observações:

- O mesmo jogo pode ser lançado em várias plataformas, portanto o número de duplicados apenas na coluna `name` é esperado.
- Os dois jogos com `name` ausente foram preenchidos com "unknown" porque não interferem no objetivo da análise.
- Poderemos visualizar os possíveis duplicados observando os pares duplicados `name`-`platform`
- Jogos com o mesmo par `name`-`platform` **são diferentes** se possuírem diferentes `year_of_release` - é uma pratica comum na insdústria de games refazer jogos de muito sucesso no passado com novas features e gráficos (Jogos Remasterizados)
- Uma verificação foi feita fora deste caderno e poderíamos ter alguns duplicados implícitos, ou seja, eram registros do mesmo jogo, mesma plataforma, mesmo ano de lançamento dividos como (All Region sales) e (weekly JP sales) e outros termos semelhantes.
- O tipo de dado da coluna está correto.

#### Deliberações:

- Podemos criar uma função auxiliar no nosso caderno para identificar e resolver os duplicados impícitos para o nosso DataFrame, condensando os dados em um único registro.

### Coluna 1: `platform`

In [ ]:
column = df["platform"]

print("Verificando a presença de duplicados:")
print(column.duplicated().sum())
print("-"*42)
print("Verificando a presença de ausentes:")
print(column.isna().sum())
print("-"*42)
# Variável Categórica
print("Valores observados:")
print(f"Total de valores diferentes: {column.nunique()}")
print(column.unique().tolist())
#print(column.value_counts())
print("-"*42)

In [ ]:
df["platform"] = df["platform"].str.upper()
print(f"Total de valores diferentes: {df["platform"].nunique()}")
print(df["platform"].unique().tolist())

#### Observações:

- A coluna `platform` não possui valores ausentes ou duplicados implícitos.
- Os valores se repetem de forma normal e já esperada, como uma coluna categórica. O tipo de dado está correto.
- A fim de facilitar qualquer filtragem futura, atribuímos a todos os registros os caracteres em 'CAPITAL LETTERS'. Essa alteração foi segura e não reduziu o total de valores diferentes. (Wii e WiiU poderiam causar alguma confusão)

#### Deliberações:

(Nada)

### Coluna 2: `year_of_release`

In [ ]:
column = df["year_of_release"]

print("Verificando a presença de duplicados:")
print(column.duplicated().sum())
print("-"*42)
print("Verificando a presença de ausentes:")
print(column.isna().sum())
print("-"*42)
# Variável Categórica
print("Valores observados:")
print(f"Total de valores diferentes: {column.nunique()}")
print(column.unique().tolist())
print(f"\nTop 5 valores mais frequentes: {column.value_counts().head()}")
print("-"*42)

Vamos olhar com mais profundidade para os valores ausentes porque precisaremos preenchê-los com um número inteiro para que possamos ajustar o tipo de dado da coluna de forma segura, e assim completar esta coluna de valores inteiros de natureza categórica.

In [ ]:
mask1 = df["year_of_release"].isna()
year_pattern = r"(19|20)\d{2}"
print(df[mask1]["platform"].value_counts())

df1 = df[mask1]
mask2 = df1["name"].str.contains(r"(19|20)\d{2}", regex=True, na=False)

df1[mask2]


In [ ]:
df["year_of_release"] = df["year_of_release"].fillna(3000)
print("Verificando a presença de ausentes:")
print(df["year_of_release"].isna().sum())

# Convertendo o tipo de dado de forma segura:
year_change = safe_series_to_int(df["year_of_release"])
df["year_of_release"] = year_change
#Verificando o tipo de dado
print(df["year_of_release"].dtype)

#### Observações:

- A coluna `year_of_release` não possui valores duplicados implícitos.
- Observamos 269 registros com ano de lançamento ausente. Rodamos um código acima em um DataFrame filtrado e observamos que estes valores ausentes estão presentes em diversas plataformas.
- Filtramos e observamos também alguns jogos que já tinham o "ano de lançamento" no próprio nome. No entando, jogos como o "Sega Rally 2006" foram lançados no ano exibido no nome enquanto outros jogos como "FIFA Soccer 2004" foram lançados no ano anterior ao exibido.
- Decidimos preencher os valores com um noov valor inteiro "absurdo" para esta categoria (ano 3000) para que não tenhamos valores ausentes por enquanto, e **saibamos exatamente quais registros estavam ausentes no início da análise**.


#### Deliberações:

- Não criaremos, ao menos por enquanto, nenhuma função que interprete nomes para preencher a coluna `year_of_release`.
- Criamos neste caderno uma função que recebe um Objeto Series e Verifica conversão segura (sem perder valores decimais). **JÁ PALICADA ACIMA.**

### Coluna 3: `genre`

In [ ]:
column = df["genre"]

print("Verificando a presença de duplicados:")
print(column.duplicated().sum())
print("-"*42)
print("Verificando a presença de ausentes:")
print(column.isna().sum())
print("-"*42)
# Variável Categórica
print("Valores observados:")
print(f"Total de valores diferentes: {column.nunique()}")
print(column.unique().tolist())
# print(column.value_counts())
print("-"*42)

In [ ]:
df["genre"] = df["genre"].str.upper()
print(f"Total de valores diferentes: {df["genre"].nunique()}")
print(df["genre"].unique().tolist())

In [ ]:
df[df["genre"].isna()]

In [ ]:
df["genre"] = df["genre"].fillna("unknown")
print(f"Nova contagem de valores ausentes: {df["genre"].isna().sum()}")

#### Observações:

- A coluna `genre` não possui valores duplicados implícitos.
- Os valores se repetem de forma normal e já esperada, como uma coluna categórica. O tipo de dado está correto.
- Os valores ausentes da coluna `genre` estavam nos mesmos registros dos ausentes da coluna `name` que já estavam preenchidos com a palavra "unknown". Seguimos o mesmo critério para este objeto series.
- A fim de facilitar qualquer filtragem futura, atribuímos a todos os registros os caracteres em 'CAPITAL LETTERS'. Essa alteração foi segura e não reduziu o total de valores diferentes.

#### Deliberações:

(Nada)

### Coluna 4: `na_sales`

In [ ]:
column = df["na_sales"]

print("Verificando a presença de duplicados:")
print(column.duplicated().sum())
print("-"*42)
print("Verificando a presença de ausentes:")
print(column.isna().sum())
print("-"*42)
# Variável Numérica
print("Valores observados:")
print(f"Total de valores diferentes: {column.nunique()}")
print("-"*42)
print(f"Média da coluna: {column.mean():.4}")
print(f"Mediana da coluna: {column.median()}")
print(f"Moda da coluna: {float(safe_mode(column.dropna()).iloc[0])}")
print(f"Máximo: {column.max():.4}")
print(f"Amplitude: {round(column.max() - column.min(), 4)}")
print(f"Desvio Padrão: {column.std():.4}")
print("-"*42)

### Coluna 5: `eu_sales`

In [ ]:
column = df["eu_sales"]

print("Verificando a presença de duplicados:")
print(column.duplicated().sum())
print("-"*42)
print("Verificando a presença de ausentes:")
print(column.isna().sum())
print("-"*42)
# Variável Numérica
print("Valores observados:")
print(f"Total de valores diferentes: {column.nunique()}")
print("-"*42)
print(f"Média da coluna: {column.mean():.4}")
print(f"Mediana da coluna: {column.median()}")
print(f"Moda da coluna: {float(safe_mode(column.dropna()).iloc[0])}")
print(f"Máximo: {column.max():.4}")
print(f"Amplitude: {round(column.max() - column.min(), 4)}")
print(f"Desvio Padrão: {column.std():.4}")
print("-"*42)

### Coluna 6: `jp_sales`

In [ ]:
column = df["jp_sales"]

print("Verificando a presença de duplicados:")
print(column.duplicated().sum())
print("-"*42)
print("Verificando a presença de ausentes:")
print(column.isna().sum())
print("-"*42)
# Variável Numérica
print("Valores observados:")
print(f"Total de valores diferentes: {column.nunique()}")
print("-"*42)
print(f"Média da coluna: {column.mean():.4}")
print(f"Mediana da coluna: {column.median()}")
print(f"Moda da coluna: {float(safe_mode(column.dropna()).iloc[0])}")
print(f"Máximo: {column.max():.4}")
print(f"Amplitude: {round(column.max() - column.min(), 4)}")
print(f"Desvio Padrão: {column.std():.4}")
print("-"*42)

### Coluna 7: `other_sales`

In [ ]:
column = df["other_sales"]

print("Verificando a presença de duplicados:")
print(column.duplicated().sum())
print("-"*42)
print("Verificando a presença de ausentes:")
print(column.isna().sum())
print("-"*42)
# Variável Numérica
print("Valores observados:")
print(f"Total de valores diferentes: {column.nunique()}")
print("-"*42)
print(f"Média da coluna: {column.mean():.4}")
print(f"Mediana da coluna: {column.median()}")
print(f"Moda da coluna: {float(safe_mode(column.dropna()).iloc[0])}")
print(f"Máximo: {column.max():.4}")
print(f"Amplitude: {round(column.max() - column.min(), 4)}")
print(f"Desvio Padrão: {column.std():.4}")
print("-"*42)

#### Observações:

- Não temos nenhuma linha com valores ausentes em dados de venda.
- Os tipos de dado das colunas está correto.
- É importante já pensar em construir uma nova feature na tabela com vendas totais.

#### Deliberações:

- Criar função de linha que será aplicada para gerar as vendas totais.

### Coluna 8: `critic_score`

In [ ]:
column = df["critic_score"]

print("Verificando a presença de duplicados:")
print(column.duplicated().sum())
print("-"*42)
print("Verificando a presença de ausentes:")
print(column.isna().sum())
print("-"*42)
# Variável Categórica
print("Valores observados:")
print(f"Total de valores diferentes: {column.nunique()}")
print("-"*42)
print(f"Média da coluna: {column.dropna().mean():.4}")
print(f"Mediana da coluna: {column.median()}")
print(f"Moda da coluna: {float(safe_mode(column.dropna()).iloc[0])}")
print(f"Mínimo: {column.min():.4}")
print(f"Máximo: {column.max():.4}")
print(f"Amplitude: {round(column.max() - column.min(), 4)}")
print(f"Desvio Padrão: {column.std():.4}")
print("-"*42)

Observando os dados acima, percebemos que é um tipo de dado float que tem um espaço amostral limitado de 0 a 100. A média com certeza está um pouco abaixo da mediana por conta dos 0,5% (48 notas) abaixo de 3 desvios-padrão dela. 

Ainda assim, podemos considerar que a distribuição das notas dadas pelos críticos não tem muitos valores atípicos, e por isso podemos preencher os valores ausentes com a média e manter o tipo de dado da coluna como float.

In [ ]:
df["critic_score"] = df["critic_score"].fillna(68.97)
print("Verificando a presença de ausentes:")
print(df["critic_score"].isna().sum())
print("-"*42)
print(f"Média da coluna: {df["critic_score"].dropna().mean():.4}")
print(f"Mediana da coluna: {df["critic_score"].median()}")
print(f"Moda da coluna: {float(safe_mode(df["critic_score"].dropna()).iloc[0])}")

#### Observações:

- Optamos por preencher os valores ausentes com a média da coluna uma vez que não tínhamos uma distribuição com muitos valores atípicos.
- Tipo de dado float foi mantido, assim como a média foi mantida. A mediana e a Moda se tornaram o mesmo valor da média.

#### Deliberações:

(nada)

### Coluna 9: `user_score`

Ao tentar começar a observar os dados de user_score, foi fácil perceber que ele continha o tipo de dados equivocado (str), mas isso se deve ao fato da presença de "tbd" como avaliação em alguns registros como veremos abaixo.

In [ ]:
df["user_score"].value_counts()

Antes de qualquer decisão, vamos observar algumas estatísticas descritivas desta coluna, excluindo momentaneamente "tbd" e valores ausentes para transformar os dados em tipo float.

In [ ]:
# Removendo os dados Ausentes
column = df["user_score"].dropna()
# Removendo os registros "tbd"
column = column[column != "tbd"]
column = column.astype(float)

print("Total de dados analisados:")
print(len(column))
print("-"*42)
print("Verificando a presença de duplicados:")
print(column.duplicated().sum())
print("-"*42)
print("Verificando a presença de ausentes:")
print(column.isna().sum())
print("-"*42)
print("Valores observados:")
print(f"Total de valores diferentes: {column.nunique()}")
print("-"*42)
print(f"Média da coluna: {column.mean():.4}")
print(f"Mediana da coluna: {column.median()}")
print(f"Mínimo: {column.min():.4}")
print(f"Máximo: {column.max():.4}")
print(f"Amplitude: {round(column.max() - column.min(), 4)}")
print(f"Desvio Padrão: {column.std():.4}")
print("-"*42)

Observando os dados acima, eles também se distribuem como era esperado entre 0 e 10, ou seja, em uma faixa limitada de valores possíveis no espaço amostral. 

A presença dos "tbd" tem o mesmo valor semântico dos valores ausentes: não houve nenhuma crítica ou nota estabelecida por um usuário que possa ser diretamente associada ao par `name - platform`.

Considerando que não temos muitos valores atípicos superiores nem inferiores (apenas 1.3% dos valores observados abaixo de 3 desvios-padrão), vamos preencher tanto os valores ausentes quanto os "tbd" com a média, e transformar todo tipo de dado da coluna em float.

In [ ]:
df["user_score"] = df["user_score"].fillna("7.125")
df["user_score"] = df["user_score"].replace("tbd","7.125")
df["user_score"] = df["user_score"].astype(float)
print("Verificando a presença de ausentes:")
print(df["user_score"].isna().sum())
print("-"*42)
print(f"Média da coluna: {df["user_score"].mean():.4}")
print(f"Mediana da coluna: {df["user_score"].median()}")
print(f"Moda da coluna: {float(safe_mode(df["user_score"].dropna()).iloc[0])}")

#### Observações:

- Optamos por preencher os valores ausentes e os valores "tbd" com a média da coluna uma vez que não tínhamos uma distribuição com muitos valores atípicos.
- Essa mudança permitiu que transformássemos os dados da coluna no tipo float, e consequentemente podendo operar com outros dados numéricos. A mediana e a Moda se tornaram o mesmo valor da média.

#### Deliberações:

- Criar uma função capaz de realizar uma média ponderada entre as avaliações dos usuários e dos críticos, podendo condensar em uma nova feature da tabela uma única nota de avaliação. 

### Coluna 10: `rating`

In [ ]:
column = df["rating"]

print("Verificando a presença de duplicados:")
print(column.duplicated().sum())
print("-"*42)
print("Verificando a presença de ausentes:")
print(column.isna().sum())
print("-"*42)
# Variável Categórica
print("Valores observados:")
print(f"Total de valores diferentes: {column.nunique()}")
print(column.unique().tolist())
print("-"*42)
print(column.value_counts())

Resumindo a tabela para um leitor possivelmente leigo no assunto:
| Classificação   | Significado     | Em português           | Indicação                                                   |
| -------- | --------------- | ---------------------- | ----------------------------------------------------------- |
| **E**    | Everyone        | Todos                  | Geralmente adequado para todas as idades                    |
| **M**    | Mature          | Maduro                 | **17 anos ou mais**                                         |
| **T**    | Teen            | Adolescente            | **13 anos ou mais**                                         |
| **E10+** | Everyone 10+    | Todos 10+              | **10 anos ou mais**                                         |
| **K-A**  | Kids to Adults  | Crianças a adultos     | Categoria antiga equivalente aproximadamente ao atual **E** |
| **AO**   | Adults Only     | Somente adultos        | **18 anos ou mais**                                         |
| **EC**   | Early Childhood | Primeira infância      | Categoria antiga destinada a crianças pequenas              |
| **RP**   | Rating Pending  | Classificação pendente | Ainda não recebeu classificação final                       |


Pesquisando um pouco sobre o os índices ESRB de avaliações dos jogos, podemos fazer duas observações relevantes para análise:
- `K-A` é uma clássificação antiga que significa "Kids to Adults", e de certa forma é implícito que seja igual à categoria `E` que significa "everyone".
-  `EC` significa "early childhood". Apesar de ser adequado a todos os públicos, a proposta do jogo é diferente. Vamos manter a sigla, mas sabendo que a classificação foi descontinuada em 2018, e no nosso dataset o jogo mais recente é de 2011.
- A categoria `RP` siginifica "rating pending". Ela aparece três vezes na nossa análise em ocorrências sem valor total de vendas significativo. Portanto, faremos o replacement dela com a mesma string que faremos o preenchimento dos valores ausentes.

In [ ]:
df["rating"] = df["rating"].replace("K-A", "E")
# Preenchendo valores ausentes
df["rating"] = df["rating"].fillna("unknown")
df["rating"] = df["rating"].replace("RP", "unknown")

print("Verificando a presença de ausentes:")
print(df["rating"].isna().sum())
print("-"*42)
# Variável Categórica
print("Valores observados:")
print(f"Total de valores diferentes: {df["rating"].nunique()}")
print(df["rating"].unique().tolist())
print("-"*42)
print(df["rating"].value_counts())

#### Observações:

- Esta coluna categórica fica então com 7 tipos diferentes de jogos, sendo o mais frequente os jogos da categoria preenchida como "unknown".

#### Deliberações:

(nada)

### Resumo do Pré-Processamento de dados

Antes de passarmos para a sessão de Análise Univariada, resolveremos algunas das deliberações dessa sessão, e imprimiremos novamente o comando `df.info()`. O que será resolvido agora:

- Na coluna `name` alguns registros que contam 

In [ ]:
df[df["name"].str.contains("sales")]

In [ ]:
# df[df["name"].str.contains("Medal of Honor: European Assault")]
df[df["name"].str.contains("NBA Live 06")]

In [ ]:
df.info()